# 07. Evaluación y Análisis de Resultados

Notebook de cierre para comparar modelos, revisar cobertura y dejar una lectura clara de fortalezas y riesgos del sistema.


In [2]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

from src.utils.recsys_utils import ensure_dir

processed_dir = ROOT / 'data' / 'processed'
tables_dir = ensure_dir(ROOT / 'reports' / 'tables')
models_dir = ensure_dir(ROOT / 'models')

In [3]:
train = pd.read_parquet(processed_dir / 'interactions_train.parquet')
test = pd.read_parquet(processed_dir / 'interactions_test.parquet')

## Consolidar métricas de modelos

Se leen los resultados producidos por notebooks previos y se arma una sola tabla comparativa.


In [4]:
metric_files = {
    'cf': tables_dir / 'cf_metrics.csv',
    'bandit': tables_dir / 'bandit_metrics.csv',
    'hybrid': tables_dir / 'hybrid_metrics.csv',
}

metrics_frames = []
for name, path in metric_files.items():
    if path.exists():
        frame = pd.read_csv(path)
        frame['source_file'] = path.name
        metrics_frames.append(frame)

if metrics_frames:
    comparison = pd.concat(metrics_frames, ignore_index=True, sort=False)
else:
    comparison = pd.DataFrame()

display(comparison)
comparison.to_csv(tables_dir / 'model_comparison.csv', index=False)


,model,precision_at_k,recall_at_k,ndcg_at_k,evaluated_users,source_file,policy,matched_events,cumulative_reward,average_reward,estimated_regret
0,cf_item_knn,0.044608,0.026028,0.053854,881.0,cf_metrics.csv,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,bandit_metrics.csv,greedy,89.0,55.0,0.617978,0.254363
2,NaN,NaN,NaN,NaN,NaN,bandit_metrics.csv,epsilon_greedy,122.0,78.0,0.639344,0.232996
3,hybrid_cf_content,0.058929,0.033140,0.068713,3864.0,hybrid_metrics.csv,NaN,NaN,NaN,NaN,NaN


## Segmentación de usuarios e ítems

Se analiza qué segmentos dominan el set de evaluación para contextualizar las métricas.


In [5]:
user_segment = (
    train.groupby('userId').size()
    .rename('n_ratings')
    .reset_index()
    .assign(segment=lambda df: pd.cut(df['n_ratings'], bins=[0, 50, 100, 250, df['n_ratings'].max()], include_lowest=True))
)

item_segment = (
    train.groupby('movieId').size()
    .rename('n_ratings')
    .reset_index()
    .assign(segment=lambda df: pd.qcut(df['n_ratings'], q=4, duplicates='drop'))
)

display(user_segment.groupby('segment').size().rename('users').reset_index())
display(item_segment.groupby('segment').size().rename('items').reset_index())


,segment,users
0,"(-0.001, 50.0]",65185
1,"(50.0, 100.0]",39587
2,"(100.0, 250.0]",39194
3,"(250.0, 13435.0]",25881


,segment,items
0,"(0.999, 26.0]",5442
1,"(26.0, 87.0]",5063
2,"(87.0, 521.0]",5205
3,"(521.0, 84724.0]",5233


## Inspección cualitativa de recomendaciones

Si existen artefactos de recomendaciones, se muestran ejemplos para revisar diversidad y coherencia.


In [6]:
qualitative_frames = []
for name, file_name in [('cf', 'cf_recommendations.parquet'), ('hybrid', 'hybrid_recommendations.parquet')]:
    path = models_dir / file_name
    if path.exists():
        frame = pd.read_parquet(path).copy()
        frame['model'] = name
        qualitative_frames.append(frame.head(10))

if qualitative_frames:
    qualitative = pd.concat(qualitative_frames, ignore_index=True)
    display(qualitative)
else:
    print('Aun no existen recomendaciones guardadas. Ejecuta primero los notebooks 05a y 05c.')


,user_idx,item_idx,score,rank,model
0,0,1291,8.195380,1,cf
1,0,573,1.018495,2,cf
2,0,576,1.002980,3,cf
3,0,1958,0.814351,4,cf
4,0,1401,0.778839,5,cf
5,0,641,0.738521,6,cf
6,0,1675,0.698859,7,cf
7,0,1804,0.657137,8,cf
8,0,286,0.648657,9,cf
9,0,1722,0.647806,10,cf
